# 05 — Intégration MCP réelle

Ce notebook teste le **vrai** serveur MCP (`mcp_server/server.py`) à travers le **vrai**
client MCP (`backend/app/tools/mcp_client.py`), en lançant un sous-processus stdio.

Contrairement à une version précédente qui simulait les outils en local, ici l'appel
traverse réellement le protocole MCP : si on modifie `mcp_server/server.py` (par exemple
la base `data/drugs.json`), le résultat de ce notebook change sans qu'on touche à ce fichier.

In [ ]:
import sys
from pathlib import Path

# Permet d'importer backend.app.tools.mcp_client depuis le dossier notebooks/
sys.path.append(str(Path("..").resolve()))

from backend.app.tools.mcp_client import get_drug_info_via_mcp, get_emergency_level_via_mcp

print("✅ Client MCP importé")

## Test 1 — get_drug_info via MCP

Chaque appel ci-dessous lance `mcp_server/server.py` en sous-processus stdio,
exécute l'outil MCP demandé, puis ferme la session.

In [ ]:
print(get_drug_info_via_mcp("paracetamol"))
print("---")
print(get_drug_info_via_mcp("ibuprofene"))
print("---")
print(get_drug_info_via_mcp("medicament_inconnu_xyz"))

## Test 2 — get_emergency_level via MCP

In [ ]:
print(get_emergency_level_via_mcp("douleur thoracique intense, difficultés à respirer"))
print("---")
print(get_emergency_level_via_mcp("fièvre élevée depuis 3 jours"))
print("---")
print(get_emergency_level_via_mcp("léger mal de tête, pas de fièvre"))

## Test 3 — utilisation telle qu'elle se produit réellement dans report_agent

`report_agent.py` n'appelle pas `mcp_client` directement : il passe par les tools
LangChain `get_drug_info` / `get_emergency_level` de `care_tools.py`, qui eux-mêmes
appellent `mcp_client`. On reproduit ici ce chemin complet.

In [ ]:
from backend.app.tools.care_tools import get_drug_info, get_emergency_level

treatment = "Paracétamol 500mg toutes les 6h"
symptoms = "fièvre élevée depuis 3 jours"

drug_name = treatment.split()[0]
drug_info = get_drug_info.invoke({"drug_name": drug_name})
urgency = get_emergency_level.invoke({"symptoms": symptoms})

print("💊 Info médicament (via tool LangChain -> mcp_client -> serveur MCP) :")
print(drug_info)
print("---")
print("🚨 Niveau urgence (via tool LangChain -> mcp_client -> serveur MCP) :")
print(urgency)
print("---")
print("✅ Intégration MCP de bout en bout fonctionnelle")